[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crunchdao/numinous/blob/master/challenge/numinous/examples/quickstart.ipynb)

![Banner](https://raw.githubusercontent.com/crunchdao/quickstarters/refs/heads/master/competitions/numinous/assets/banner.webp)

# Numinous — Binary Event Forecasting

Predict the probability that real-world events resolve **"Yes"**.

Events are sourced from prediction markets and cover politics, crypto, sports, weather, and more.

## Scoring

Predictions are evaluated with the **Brier score**:

$$\text{Brier} = (p - o)^2$$

where $p$ is your predicted probability and $o \in \{0, 1\}$ is the actual outcome. **Lower is better.**

| Score | Meaning |
|-------|---------|
| 0.00  | Perfect |
| 0.25  | Uninformative (always 0.5) |
| 1.00  | Worst possible |

Predictions are clipped to $[0.01, 0.99]$. Missing predictions are scored as 0.25.

## Setup

In [ ]:
%pip install crunch-numinous

In [2]:
from numinous.tracker import TrackerBase

## Your Model

Subclass `TrackerBase` and implement `_predict()`.

You receive event data via `feed_update()` and return a probability via `predict()`:

```
Input:  {"event_id": "...", "title": "Will X happen?", "yes_price": 0.65, ...}
Output: {"event_id": "...", "prediction": 0.72}
```

This example follows the market price — a surprisingly strong baseline that's hard to beat.

### How top-scoring models work

The best models in this competition are **LLM-powered forecasting agents**. They typically:

1. **Classify the event** by category (politics, crypto, sports, weather, earnings, etc.)
2. **Gather context** — fetch related market data from Polymarket, run web searches for recent news
3. **Prompt an LLM** with the event details + gathered context and ask it to estimate P(Yes)
4. **Parse and calibrate** the LLM's probability output

Events span very different domains. Top models use specialized prompts and data sources per category:

| Category | Example event | Context sources |
|----------|--------------|-----------------|
| Crypto | "Will BTC exceed $100k?" | Price feeds, on-chain data |
| Politics | "Will the Fed cut rates?" | News, policy analysis |
| Weather | "Will it rain in NYC?" | Weather APIs, forecasts |
| Sports | "Will Team X win?" | Stats, odds, schedules |
| Earnings | "Will AAPL beat Q2 estimates?" | Financial data, analyst consensus |

This starter uses `yes_price` directly. To compete, you'll want to add LLM reasoning — the event `title`, `description`, and `metadata` fields give you everything you need to build a prompt.

In [5]:
class MarketTracker(TrackerBase):
    """Returns the current market yes_price as the prediction."""

    def _predict(
        self,
        subject: str,
        resolve_horizon_seconds: int,
        step_seconds: int
    ):
        data = self._get_data(subject)
        if not isinstance(data, dict):
            return {
                "event_id": subject,
                "prediction": 0.5,
            }

        event_id = data.get("event_id", subject)
        prediction = max(0.0, min(1.0, float(data.get("yes_price", 0.5))))

        return {
            "event_id": event_id,
            "prediction": prediction,
        }

## Test Locally

Feed sample events and check predictions.

In [ ]:
SAMPLE_EVENTS = [
    {"event_id": "evt-btc-100k", "title": "Will BTC exceed $100k by end of March?",
     "yes_price": 0.65, "cutoff": "2026-04-01T00:00:00Z", "source": "polymarket",
     "description": "", "volume_24h": 150000.0, "metadata": {}},
    {"event_id": "evt-rain-nyc", "title": "Will it rain in NYC tomorrow?",
     "yes_price": 0.80, "cutoff": "2026-03-25T00:00:00Z", "source": "polymarket",
     "description": "", "volume_24h": 5000.0, "metadata": {}},
    {"event_id": "evt-fed-rate", "title": "Will the Fed cut rates in April?",
     "yes_price": 0.35, "cutoff": "2026-04-30T00:00:00Z", "source": "polymarket",
     "description": "", "volume_24h": 500000.0, "metadata": {}},
]

OUTCOMES = {"evt-btc-100k": 1, "evt-rain-nyc": 1, "evt-fed-rate": 0}

model = MarketTracker()
total_brier = 0.0

print(f"{'Event':50s} {'Pred':>6s} {'Outcome':>8s} {'Brier':>8s}")
print("-" * 76)

for event in SAMPLE_EVENTS:
    model.feed_update(event)
    result = model.predict(event["event_id"], resolve_horizon_seconds=3600, step_seconds=1)

    p = max(0.01, min(0.99, result["prediction"]))
    o = OUTCOMES[event["event_id"]]
    brier = (p - o) ** 2
    total_brier += brier

    print(f"{event['title'][:50]:50s} {p:6.2f} {'Yes' if o else 'No':>8s} {brier:8.4f}")

print("-" * 76)
print(f"{'Average Brier':50s} {total_brier / len(SAMPLE_EVENTS):>22.4f}")
print(f"{'Baseline (always 0.5)':50s} {'0.2500':>22s}")

# Submit your Notebook

To submit your work, you must:
1. Download your Notebook from Colab
2. Upload it to the platform
3. Deploy it in real condition

### >> https://hub.crunchdao.com/competitions/numinous/submit/notebook

The platform will find your `TrackerBase` subclass automatically.

![Download and Submit Notebook](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/animations/download-and-submit-notebook-deployment.gif)